# 14. Ensemble và Test-Time Augmentation (TTA)
Nhánh GitHub: `14-ensemble-tta`

Không huấn luyện lại. **TTA:** dịch vòng clip 5 lần (0, 20, 40, 60, 80%) rồi lấy trung bình xác suất. **Ensemble:** trung bình xác suất của nhiều cấu hình trong cùng một fold (mô hình ở fold k chưa thấy fold k nên không rò rỉ). Cấu hình tốt nhất chọn theo Macro-F1 **validation**, không chọn theo test.

In [1]:
!pip -q install soundfile
from google.colab import drive
drive.mount('/content/drive')
import sys
sys.path.append('/content/drive/MyDrive/ESC50')
from esc_lib import *
setup()

Mounted at /content/drive
GPU: Tesla T4 | VRAM 15.6 GB | torch 2.14.1+cu130 | CUDA 13.0
RAM trống 11.7 GB | ổ đĩa trống 120.8 GB
Mixup: BẬT | quick: False | kết quả lưu ở: /content/drive/MyDrive/ESC50/results_mixup


In [2]:
meta, X, X_DEV, y, folds = load_data()

  0%|          | 0/2000 [00:00<?, ?it/s]

X: (2000, 110250) | y: (2000,) | nằm trên: cuda:0


In [4]:
ENSEMBLE_EXPS = None
runs, rob = load_results()
FOLDS_DONE = sorted(runs["fold"].unique())

acc_rank = runs.groupby("exp")["val_f1"].mean().to_dict()
cands = [e for e in sorted(acc_rank) if all(find_ckpt(e, f) is not None for f in FOLDS_DONE)]
print("Cấu hình dùng được:", cands)

P, Y = {}, {}
for exp in cands:
    for fold in FOLDS_DONE:
        te_idx = get_split(fold)[2]
        model, front, mean, std, src = load_member(exp, fold)
        P[exp, fold, "plain"], Y[fold] = predict_proba(model, front, mean, std, src, te_idx)
        P[exp, fold, "tta"], _ = predict_proba(model, front, mean, std, src, te_idx, shifts=TTA_SHIFTS)
        del model, front
    print(f"  {exp}: xong")
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()


def eval_sets(prob_fn):
    a, f = [], []
    for fold in FOLDS_DONE:
        acc, f1 = scores(Y[fold], prob_fn(fold).argmax(axis=1))
        a.append(acc)
        f.append(f1)
    return np.array(a), np.array(f)


top3 = sorted(cands, key=acc_rank.get, reverse=True)[:3]
ens_sets = {"Ensemble top-3": top3, "Ensemble tất cả": cands}
if ENSEMBLE_EXPS:
    ens_sets = {"Ensemble tự chọn": list(ENSEMBLE_EXPS), **ens_sets}

rows = []
for exp in cands:
    for mode in ("plain", "tta"):
        a, f = eval_sets(lambda fold: P[exp, fold, mode])
        rows.append(dict(name=exp, members=exp, tta=mode == "tta", acc=a, f1=f))
for name, members in ens_sets.items():
    for mode in ("plain", "tta"):
        a, f = eval_sets(lambda fold: sum(P[e, fold, mode] for e in members) / len(members))
        rows.append(dict(name=name, members="+".join(members), tta=mode == "tta", acc=a, f1=f))

rows.sort(key=lambda r: -r["acc"].mean())
ens_table = pd.DataFrame({
    "Cấu hình": [r["name"] for r in rows],
    "Thành phần": [r["members"] for r in rows],
    "TTA": ["Có" if r["tta"] else "Không" for r in rows],
    "Accuracy (%)": [f"{r['acc'].mean() * 100:.2f} ± {r['acc'].std(ddof=1) * 100:.2f}" for r in rows],
    "Macro-F1 (%)": [f"{r['f1'].mean() * 100:.2f} ± {r['f1'].std(ddof=1) * 100:.2f}" for r in rows],
})
display(ens_table)
ens_table.to_csv(RESULT_DIR / "ensemble_tta.csv", index=False)

val_best = max(cands, key=acc_rank.get)
best = next(r for r in rows if r["name"] == val_best and r["tta"])
best_single = next(r for r in rows if r["name"] == val_best and not r["tta"])
test_top = rows[0]
print(f"\nChọn theo validation: {val_best} (Macro-F1 validation trung bình {acc_rank[val_best] * 100:.2f}%), dùng TTA")
print(f"  Kết quả test: {best['acc'].mean() * 100:.2f}% (có TTA) | {best_single['acc'].mean() * 100:.2f}% (không TTA) | "
      f"TTA {100 * (best['acc'].mean() - best_single['acc'].mean()):+.2f} điểm")
print(f"  Dòng cao nhất trong bảng test: {test_top['name']} ({test_top['members']}), TTA={'Có' if test_top['tta'] else 'Không'}: "
      f"{test_top['acc'].mean() * 100:.2f}% (chỉ để tham khảo, không dùng để chọn)")

_vf = runs[runs["exp"] == val_best].set_index("fold")["val_f1"]
best_fold = int(_vf.idxmax()) if len(_vf) else FOLDS_DONE[0]
choice = dict(val_best=val_best, members=best["members"].split("+"), best_fold=best_fold,
              tta_shifts=list(TTA_SHIFTS), val_f1=float(acc_rank[val_best]))
(RESULT_DIR / "ensemble_choice.json").write_text(json.dumps(choice, indent=2))
print("Đã lưu lựa chọn:", choice)

Cấu hình dùng được: ['E1', 'E2', 'E3', 'E4', 'E5', 'E6', 'E7', 'E8', 'E9']
  E1: xong
  E2: xong
  E3: xong
  E4: xong
  E5: xong
  E6: xong
  E7: xong
  E8: xong
  E9: xong


,Cấu hình,Thành phần,TTA,Accuracy (%),Macro-F1 (%)
0,E9,E9,Có,94.15 ± 2.13,93.91 ± 2.31
1,E9,E9,Không,93.80 ± 2.35,93.57 ± 2.52
2,Ensemble top-3,E9+E3+E4,Có,92.20 ± 1.47,91.83 ± 1.80
3,Ensemble top-3,E9+E3+E4,Không,91.85 ± 1.34,91.47 ± 1.62
4,Ensemble tất cả,E1+E2+E3+E4+E5+E6+E7+E8+E9,Có,89.20 ± 1.23,88.71 ± 1.56
5,Ensemble tất cả,E1+E2+E3+E4+E5+E6+E7+E8+E9,Không,88.95 ± 0.94,88.55 ± 1.19
6,E2,E2,Có,84.60 ± 1.69,83.98 ± 1.75
7,E2,E2,Không,84.55 ± 2.09,83.95 ± 2.23
8,E3,E3,Có,83.95 ± 2.40,83.32 ± 2.99
9,E4,E4,Có,83.25 ± 1.72,82.72 ± 1.84



Chọn theo validation: E9 (Macro-F1 validation trung bình 94.68%), dùng TTA
  Kết quả test: 94.15% (có TTA) | 93.80% (không TTA) | TTA +0.35 điểm
  Dòng cao nhất trong bảng test: E9 (E9), TTA=Có: 94.15% (chỉ để tham khảo, không dùng để chọn)
Đã lưu lựa chọn: {'val_best': 'E9', 'members': ['E9'], 'best_fold': 1, 'tta_shifts': [0.0, 0.2, 0.4, 0.6, 0.8], 'val_f1': 0.9467670167122181}
